# Bulls & Cows — Cold SFT Warmup

Fine-tune **Qwen3-4B** on reasoning chains (CoT) before RL (GRPO) training.

The SFT dataset contains extracted reasoning chains with correct answers
in `<think>...</think><answer>...</answer>` format.

**Sections:**
1. [Environment Setup](#Environment-Setup)
2. [Model Setup](#Model-Setup)
3. [Data Preparation](#Data-Preparation)
4. [Training](#Training)
5. [Inference Test](#Inference-Test)
6. [Save Model](#Save-Model)

<a name="Environment-Setup"></a>
## 1. Environment Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2,3"
os.environ["CUDA_HOME"] = os.path.expanduser("~/venvs/gpu")
os.environ["CPATH"] = ":".join([
    os.path.expanduser("~/venvs/gpu/include"),
    os.path.expanduser("~/venvs/gpu/targets/x86_64-linux/include"),
])
os.environ["LIBRARY_PATH"] = ":".join([
    os.path.expanduser("~/venvs/gpu/lib"),
    os.path.expanduser("~/venvs/gpu/lib/stubs"),
])

<a name="Model-Setup"></a>
## 2. Model Setup

Load the same base model as GRPO: Qwen3-4B with 4-bit quantization and LoRA.

In [2]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch

max_seq_length = 8192
lora_rank = 64

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B",
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    fast_inference=False,  # No vLLM needed for SFT
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.15.1.
   \\   /|    NVIDIA H200. Num GPUs = 3. Max memory: 139.811 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth 2026.2.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


<a name="Data-Preparation"></a>
## 3. Data Preparation

Load the SFT CoT dataset and convert to the chat template format.

In [3]:
import json
from datasets import Dataset

SFT_DATA_PATH = "data/sft_cot_dataset_difficulty_1.jsonl"

# Load the dataset
records = []
with open(SFT_DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print(f"Loaded {len(records)} SFT examples")
print(f"Sample messages[0] (system): {records[0]['messages'][0]['content'][:80]}...")
print(f"Sample messages[2] (assistant): {records[0]['messages'][2]['content'][:200]}...")

Loaded 811 SFT examples
Sample messages[0] (system): Respond in the following format:
<think>
...
</think>
<answer>
...
</answer>...
Sample messages[2] (assistant): <think>
Okay, let's try to solve this Bulls and Cows puzzle step by step. So, we have a 3-digit secret number with all unique digits, and we need to figure it out based on the given guesses and their ...


In [4]:
# Convert messages to text using the tokenizer's chat template
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(records)
dataset = dataset.map(format_chat, num_proc=4)

print(f"\nDataset: {dataset}")
print(f"\nSample text (first 500 chars):")
print(dataset[0]["text"][:500])

Map (num_proc=4):   0%|          | 0/811 [00:00<?, ? examples/s]


Dataset: Dataset({
    features: ['messages', 'answer', 'difficulty', 'text'],
    num_rows: 811
})

Sample text (first 500 chars):
<|im_start|>system
Respond in the following format:
<think>
...
</think>
<answer>
...
</answer><|im_end|>
<|im_start|>user
Bulls and Cows is a code-breaking game. A secret number of 3 digits has been chosen. Every digit in the secret number is unique (no repeated digits), and digits are taken from 0-9. The number MAY start with 0.

You are given a series of guesses along with their scores:
- A "bull" means a digit in the guess is correct AND in the correct position.
- A "cow" means a digit in th


In [5]:
# Analyze token lengths to choose max_seq_length for training
import numpy as np

lengths = []
for ex in dataset:
    tokens = tokenizer.encode(ex["text"], add_special_tokens=False)
    lengths.append(len(tokens))

lengths = np.array(lengths)
print(f"Token length stats:")
print(f"  Mean:   {lengths.mean():.0f}")
print(f"  Median: {np.median(lengths):.0f}")
print(f"  P95:    {np.percentile(lengths, 95):.0f}")
print(f"  P99:    {np.percentile(lengths, 99):.0f}")
print(f"  Max:    {lengths.max()}")
print(f"\nExamples > 8192 tokens: {(lengths > 8192).sum()}")

Token length stats:
  Mean:   5448
  Median: 5467
  P95:    6976
  P99:    7213
  Max:    7280

Examples > 8192 tokens: 0


<a name="Training"></a>
## 4. Training

Train with SFTTrainer from `trl`. We do a short warmup (1-2 epochs)
to teach the model the reasoning format before switching to RL.

In [6]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    # Dataset
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=True,  # Pack short sequences together for efficiency

    # Training hyperparams
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch size = 16
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    seed=3407,

    # Logging & saving
    logging_steps=5,
    save_steps=50,
    save_total_limit=3,
    output_dir="outputs/sft_warmup",
    report_to="none",
)

In [7]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=sft_config,
)

print(f"Training samples: {len(dataset)}")
print(f"Estimated steps: {trainer.state.max_steps if hasattr(trainer.state, 'max_steps') else 'N/A'}")

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/811 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=64):   0%|          | 0/811 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!
Training samples: 811
Estimated steps: 0


In [8]:
# Train!
train_result = trainer.train()
print(f"\nTraining complete!")
print(f"  Total steps: {trainer.state.global_step}")
print(f"  Final loss:  {train_result.training_loss:.4f}")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 766 | Num Epochs = 3 | Total steps = 144
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 132,120,576 of 4,154,588,672 (3.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,0.826300
10,0.590800
15,0.527400
20,0.456600
25,0.412100
30,0.388400
35,0.374000
40,0.366800
45,0.357800
50,0.351800



Training complete!
  Total steps: 144
  Final loss:  0.3764


<a name="Inference-Test"></a>
## 5. Inference Test

Quick test to see if the model learned the reasoning format.

In [ ]:
# Switch model to inference mode
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = """
Respond in the following format:
<think>
...
</think>
<answer>
...
</answer>
"""

# Pick a sample from the dataset
test_example = records[0]
test_question = test_example["messages"][1]["content"]
gold_answer = test_example.get("answer", "N/A")

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_question},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=4096,
        temperature=0.6,
        top_p=0.95,
        do_sample=True,
    )

response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

print(f"Gold answer: {gold_answer}")
print(f"---")
print(f"Model response (first 2000 chars):")
print(response[:2000])

In [ ]:
# Test on a few more examples
import random
random.seed(42)

test_indices = random.sample(range(len(records)), min(5, len(records)))
correct = 0

for idx in test_indices:
    ex = records[idx]
    q = ex["messages"][1]["content"]
    gold = ex.get("answer", "")

    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": q},
    ]

    inp = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            input_ids=inp,
            max_new_tokens=4096,
            temperature=0.6,
            top_p=0.95,
            do_sample=True,
        )

    resp = tokenizer.decode(out[0][inp.shape[-1]:], skip_special_tokens=True)

    # Extract answer
    import re
    extracted = ""
    if "<answer>" in resp and "</answer>" in resp:
        ans_block = resp.split("<answer>")[-1].split("</answer>")[0].strip()
        digits = re.findall(r"\d+", ans_block)
        if digits:
            extracted = digits[-1]

    is_correct = extracted == gold
    if is_correct:
        correct += 1

    print(f"[{'✓' if is_correct else '✗'}] Gold: {gold} | Extracted: {extracted}")

print(f"\nAccuracy: {correct}/{len(test_indices)} = {correct/len(test_indices)*100:.0f}%")

<a name="Save-Model"></a>
## 6. Save Model

Save the SFT-warmed LoRA adapter. This will be used as the starting point for GRPO training.

In [ ]:
# Save LoRA adapter
SAVE_PATH = "outputs/sft_warmup_lora"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"\n✅ SFT-warmed LoRA adapter saved to: {SAVE_PATH}")
print(f"\nTo use in GRPO training, load with:")
print(f"  model_name = '{SAVE_PATH}'")

In [ ]:
# Optional: merge LoRA into base model and save full model
# MERGED_PATH = "outputs/sft_warmup_merged"
# model.save_pretrained_merged(MERGED_PATH, tokenizer)
# print(f"Merged model saved to: {MERGED_PATH}")